# Stage 2 – Data Preprocessing (V6 v2, 2019 dates)

Prepares and checks the two telemetry datasets before relationship mapping (Stage 3):

- **host** = host / process telemetry – Component 1 output (`C1_Output_for_C3_V6_v2_2019.csv`, 138,945 events, 26 columns)
- **iot** = IoT network flows – Component 2 output (`C2_Output_for_C3_V6_v2_2019.csv`, 59,415 flows, 22 columns)

These are the **V6 v2 files** (timestamps in 2019). Both files are **emulated component outputs** (generated to match the
Component 1 / Component 2 interface contracts), not real captured records. Part of each file is synthetic follow-up activity
that was added to the small scenarios so that every scenario has enough structure; this is checked in Steps 6 and 6b.

This notebook does **not change any value**. It checks the data, explains every placeholder, reviews every column
(role, redundancy), checks the consistency of processes, keys and order, tests for leakage, and records the preprocessing decisions.

Pipeline: load → quality checks → placeholders (`-` / blank) → column review → consistency → leakage →
decisions and hand-off to Stage 3.

**Output:** `column_roles.csv` (role of every column, used by Stage 4 when the graph features are built).

## Where each step of the data-preparation pipeline happens

| # | Step | Where it is done | Status |
|---|---|---|---|
| 1 | Data collection | emulated by the generator (the two V6 v2 files) | done |
| 2 | Data understanding | Step 2 (rows, columns, types, labels, event types) | this notebook |
| 3 | Data cleaning | Steps 3 and 4 (duplicates, invalid values, placeholders explained; nothing needs removing or imputing) | this notebook |
| 4 | Data transformation (encoding, scaling) | decided in Steps 4 and 5, **applied in Stage 4** because each graph is built from its own scenario and the scaler is fitted on the training scenarios only | Stage 4 |
| 5 | Feature engineering / selection | column roles (Step 5a) and redundancy review (Step 5b) here; the actual feature vectors are built in **Stage 4** | partly here |
| 6 | Dataset splitting | by **scenario**, not by row, so it needs the scenarios first: **Stage 3b**, applied at the start of Stage 4 (70 / 15 / 15 of the normal scenarios, all attack scenarios in the test set) | Stage 3b / 4 |
| 7 | Data validation (quality and leakage) | quality in Step 3, consistency in Step 6, leakage in Step 6b; link and scenario checks in Stage 3 / 3b | this notebook + Stage 3b |
| 8 | Save processed data | the two V6 v2 files are the final input (unchanged by this notebook); `column_roles.csv` is saved in Step 5a; the graph files are saved at the end of Stage 4 | this notebook + Stage 4 |

In [1]:
import sys
import ipaddress
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 90)

print("Libraries loaded successfully")
print("python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__)

Libraries loaded successfully
python 3.12.4 | pandas 3.0.6 | numpy 2.5.3


## Step 1: Locate the data

- **Colab:** the cell tries to mount Google Drive and looks in `MyDrive/GNN/`; if the mount fails it looks in
  `/content` (upload the two CSV files there).
- **Local / VS Code:** start Jupyter from the project folder – the files are found in `D:/GNN`, `.` or `..`.

The files are expected under the names `C1_Output_for_C3_V6_v2_2019.csv` (host) and `C2_Output_for_C3_V6_v2_2019.csv` (iot).
If you use other names, change the two lists `HOST_NAMES` and `IOT_NAMES` in the next cell.

Open the CSVs only in Colab / VS Code: re-saving them in Excel rewrites the `DD/MM/YYYY` timestamps
(day and month can be swapped and the seconds are dropped).

In [2]:
from pathlib import Path

HOST_NAMES = ["C1_Output_for_C3_V6_v2_2019.csv"]
IOT_NAMES = ["C2_Output_for_C3_V6_v2_2019.csv"]

try:                                     # only works inside Colab
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive not mounted (" + type(e).__name__ + ") - looking in local / session folders")

DIRS = ["/content/drive/MyDrive/GNN", "/content", ".", "..", "D:/GNN"]

def find(names):
    for d in DIRS:
        for n in names:
            if (Path(d) / n).is_file():
                return d, n
    return None, None

DATA_DIR, HOST_FILE = find(HOST_NAMES)
_, IOT_FILE = find(IOT_NAMES)
if HOST_FILE is None or IOT_FILE is None:
    raise FileNotFoundError(f"host / iot csv not found in {DIRS}")
print("Data folder :", Path(DATA_DIR).resolve())
print("Host file   :", HOST_FILE)
print("IoT file    :", IOT_FILE)

Drive not mounted (ModuleNotFoundError) - looking in local / session folders
Data folder : D:\GNN
Host file   : C1_Output_for_C3_V6_v2_2019.csv
IoT file    : C2_Output_for_C3_V6_v2_2019.csv


## Step 2: Load the data

Both files are read as **text** (`dtype=str`, `keep_default_na=False`) so that no value is silently converted and the
placeholders `-` and blank are kept exactly as they are. The column lists below are the expected layout.

In [3]:
# ==========================================
# STEP 2: LOAD AND BASIC STRUCTURE
# ==========================================
HOST_COLS = ["timestamp", "host_id", "host_ip", "process_id", "parent_process_id", "process_name", "event_type",
             "trace_mode", "trace_source", "src_ip", "dst_ip", "src_port", "dst_port", "protocol", "syscall_name",
             "syscall_result", "process_state", "memory_alloc_bytes", "file_operation", "threat_confidence_score",
             "threat_state", "cpu_usage", "disk_read_bytes", "disk_write_bytes", "label", "type"]
IOT_COLS = ["timestamp", "iot_device_id", "src_ip", "dst_ip", "src_port", "dst_port", "protocol", "service",
            "flow_duration", "src_bytes", "dst_bytes", "src_packets", "dst_packets", "iat_mean", "iat_std",
            "packet_size_mean", "packet_size_std", "connection_rate", "confidence_score", "label",
            "predicted_class", "attack_type"]

host = pd.read_csv(Path(DATA_DIR) / HOST_FILE, dtype=str, keep_default_na=False)
iot = pd.read_csv(Path(DATA_DIR) / IOT_FILE, dtype=str, keep_default_na=False)

print("host shape:", host.shape, "| columns exactly the expected 26:", list(host.columns) == HOST_COLS)
print("iot  shape:", iot.shape, "| columns exactly the expected 22:", list(iot.columns) == IOT_COLS)

FMT = "%d/%m/%Y %H:%M:%S"
host_ts = pd.to_datetime(host["timestamp"], format=FMT)      # strict: raises on any bad value
iot_ts = pd.to_datetime(iot["timestamp"], format=FMT)
print("\nhost time range:", host_ts.min(), "->", host_ts.max())
print("iot  time range:", iot_ts.min(), "->", iot_ts.max())
print("timestamps sorted: host", host_ts.is_monotonic_increasing, "| iot", iot_ts.is_monotonic_increasing)

print("\nhost event types:")
display(host["event_type"].value_counts().to_frame("events"))
print("class balance (label 0 = normal, 1 = attack):")
display(pd.DataFrame({"host": host["label"].value_counts(), "iot": iot["label"].value_counts()}))
display(host.head(3))
display(iot.head(3))

host shape: (138945, 26) | columns exactly the expected 26: True
iot  shape: (59415, 22) | columns exactly the expected 22: True



host time range: 2019-01-04 00:06:37 -> 2019-11-21 23:43:08
iot  time range: 2019-01-04 00:04:10 -> 2019-11-21 23:44:39
timestamps sorted: host True | iot True

host event types:


,events
event_type,
file_event,40555
network_accept,38583
memory_event,21212
network_connect,19395
process_start,19200


class balance (label 0 = normal, 1 = attack):


,host,iot
label,,
0,120796,49574
1,18149,9841


,timestamp,host_id,host_ip,process_id,parent_process_id,process_name,event_type,trace_mode,trace_source,src_ip,dst_ip,src_port,dst_port,protocol,syscall_name,syscall_result,process_state,memory_alloc_bytes,file_operation,threat_confidence_score,threat_state,cpu_usage,disk_read_bytes,disk_write_bytes,label,type
0,04/01/2019 00:06:37,web-edge-01,10.20.1.11,2004,2000,nginx,network_accept,light,xdp,203.0.113.7,10.20.1.11,5160,443,TCP,accept,success,sleeping,0,-,0.2251,normal,19.45,0,0,0,normal
1,04/01/2019 00:07:57,web-edge-02,10.20.1.12,30792,30791,curl,file_event,light,kprobe,-,-,,,-,create,success,sleeping,0,create,0.2033,normal,10.89,0,79833,0,normal
2,04/01/2019 00:10:17,ops-host-01,10.20.9.5,9558,9557,curl,process_start,light,tracepoint,-,-,,,-,execve,success,running,0,-,0.1218,normal,19.9,0,0,0,normal


,timestamp,iot_device_id,src_ip,dst_ip,src_port,dst_port,protocol,service,flow_duration,src_bytes,dst_bytes,src_packets,dst_packets,iat_mean,iat_std,packet_size_mean,packet_size_std,connection_rate,confidence_score,label,predicted_class,attack_type
0,04/01/2019 00:04:10,plc-line-b,10.30.3.32,10.30.4.42,61154,53,TCP,DNS,0.092,208,158,1,1,0.09434,0.03996,183.0,59.24,1.0,0.7407,0,normal,normal
1,04/01/2019 00:16:43,meter-floor2,10.30.2.23,10.30.1.12,29085,5683,UDP,CoAP,0.0777,52,92,1,1,0.08684,0.03905,72.0,7.06,1.0,0.8488,0,normal,normal
2,04/01/2019 00:20:13,badge-reader-01,10.30.5.51,198.51.100.3,1876,8443,TCP,HTTPS,1.8057,60681,21641,50,40,0.01766,0.00535,914.69,80.4,1.0,0.8657,0,normal,normal


**Result:** both files load with exactly the expected layout – **138,945 host events × 26 columns** and
**59,415 IoT flows × 22 columns** – and every timestamp parses with the `DD/MM/YYYY HH:MM:SS` format (4 Jan – 21 Nov
2019, in time order). Normal records are the clear majority: 120,796 normal / 18,149 attack host events (13.1 % attack)
and 49,574 normal / 9,841 attack flows (16.6 % attack). The host file has five event types: `file_event`,
`network_accept`, `memory_event`, `network_connect` and `process_start`.

## Step 3: Quality checks

Standard data-quality checks, all read from the text values: duplicates, valid IP addresses, port ranges, score
ranges, numeric columns that really parse as numbers, and valid label / category values. The attack taxonomy is the
TON_IoT category set; `lateral_movement` is **not** a row-level category (it is a connected-path property that
Component 3 itself must discover), and there is **no** `scenario_id` column.

In [4]:
# ==========================================
# STEP 3: QUALITY CHECKS
# ==========================================
def bad_ips(s):
    bad = 0
    for v in s.unique():
        if v in ("-", ""):
            continue
        try:
            ipaddress.ip_address(v)
        except ValueError:
            bad += 1
    return bad

def num(df, c):
    return pd.to_numeric(df[c].replace("", np.nan), errors="coerce")

HOST_NUM = ["process_id", "parent_process_id", "memory_alloc_bytes", "threat_confidence_score", "cpu_usage",
            "disk_read_bytes", "disk_write_bytes", "label"]
IOT_NUM = ["src_port", "dst_port", "flow_duration", "src_bytes", "dst_bytes", "src_packets", "dst_packets",
           "iat_mean", "iat_std", "packet_size_mean", "packet_size_std", "connection_rate", "confidence_score", "label"]

checks = {}
checks["duplicate rows (host / iot)"] = (int(host.duplicated().sum()), int(iot.duplicated().sum()))
checks["invalid IP strings (host / iot)"] = (bad_ips(host["src_ip"]) + bad_ips(host["dst_ip"]),
                                              bad_ips(iot["src_ip"]) + bad_ips(iot["dst_ip"]))
checks["numeric columns that fail to parse (host / iot)"] = (
    sum(int(num(host, c).isna().sum()) for c in HOST_NUM), sum(int(num(iot, c).isna().sum()) for c in IOT_NUM))
checks["negative numbers (host / iot)"] = (
    sum(int((num(host, c) < 0).sum()) for c in HOST_NUM), sum(int((num(iot, c) < 0).sum()) for c in IOT_NUM))
hp = pd.concat([num(host, "src_port"), num(host, "dst_port")]).dropna()
ip_ = pd.concat([num(iot, "src_port"), num(iot, "dst_port")])
checks["ports outside 0-65535 (host / iot)"] = (int(((hp < 0) | (hp > 65535)).sum()), int(((ip_ < 0) | (ip_ > 65535)).sum()))
checks["scores outside [0,1] (host / iot)"] = (int(((num(host, "threat_confidence_score") < 0) | (num(host, "threat_confidence_score") > 1)).sum()),
                                                int(((num(iot, "confidence_score") < 0) | (num(iot, "confidence_score") > 1)).sum()))
checks["label values other than 0/1 (host / iot)"] = (int((~host["label"].isin(["0", "1"])).sum()), int((~iot["label"].isin(["0", "1"])).sum()))

# IoT arithmetic: mean packet size = total bytes / total packets
tot_b = num(iot, "src_bytes") + num(iot, "dst_bytes")
tot_p = num(iot, "src_packets") + num(iot, "dst_packets")
checks["iot packet_size_mean vs bytes/packets, max abs difference"] = round(float((num(iot, "packet_size_mean") - tot_b / tot_p).abs().max()), 4)

for k, v in checks.items():
    print(f"{k:58s}: {v}")

print("\nhost.type values      :", sorted(host["type"].unique()))
print("iot.attack_type values:", sorted(iot["attack_type"].unique()))
banned = [c for c in ["lateral_movement", "unauthorized_access", "scan"]
          if (host["type"] == c).any() or (iot["attack_type"] == c).any()]
print("banned categories present:", banned or "none")
print("scenario_id column present:", "scenario_id" in host.columns or "scenario_id" in iot.columns)
print("normal  <-> type == 'normal'        :", bool(((host["label"] == "0") == (host["type"] == "normal")).all()))
print("host.threat_state values:", sorted(host["threat_state"].unique()), "| iot.predicted_class values:", sorted(iot["predicted_class"].unique()))

duplicate rows (host / iot)                               : (0, 0)
invalid IP strings (host / iot)                           : (0, 0)
numeric columns that fail to parse (host / iot)           : (0, 0)
negative numbers (host / iot)                             : (0, 0)
ports outside 0-65535 (host / iot)                        : (0, 0)
scores outside [0,1] (host / iot)                         : (0, 0)
label values other than 0/1 (host / iot)                  : (0, 0)
iot packet_size_mean vs bytes/packets, max abs difference : 0.005

host.type values      : ['backdoor', 'dos', 'injection', 'normal', 'password', 'ransomware', 'scanning', 'xss']
iot.attack_type values: ['backdoor', 'ddos', 'dos', 'injection', 'mitm', 'normal', 'password', 'ransomware', 'scanning', 'xss']
banned categories present: none
scenario_id column present: False
normal  <-> type == 'normal'        : True
host.threat_state values: ['normal', 'suspicious'] | iot.predicted_class values: ['attack', 'normal']


**Result:** no duplicate rows, no invalid IP addresses, every port inside 0 – 65 535, every score inside [0, 1], every
numeric column parses and none is negative, and `label` is only ever 0 or 1. The IoT arithmetic is consistent: the
mean packet size equals total bytes ÷ total packets (largest difference 0.005, which is rounding). The categories are
exactly the TON_IoT-aligned set – 7 attack types plus `normal` in the host `type` column and 9 plus `normal` in the
IoT `attack_type` column – with no `lateral_movement`, `unauthorized_access` or `scan`, and no `scenario_id`. Host
`type` is `normal` exactly when `label` is 0, including on the appended rows. (The IoT `attack_type` is Component 2's
**predicted** category, so it is not equal to the label by design.)

## Step 4: Placeholders – `-` and blank are not missing data

The host file contains `-` (text fields) and blank cells (port fields). They mean **"does not apply to this event"**,
not "value lost": a `memory_event` has no IP address, a `file_event` has no port. The check below shows that the
placeholders follow the event type exactly. They must **never be filled with invented values** (no imputation, no port 0).

In [5]:
# ==========================================
# STEP 4: PLACEHOLDERS BY EVENT TYPE
# ==========================================
share_na = pd.DataFrame({
    c: host.groupby("event_type")[c].apply(lambda s: ((s == "-") | (s == "")).mean())
    for c in ["src_ip", "dst_ip", "src_port", "dst_port", "protocol", "file_operation"]}).round(3)
share_na.insert(0, "events", host["event_type"].value_counts())
print("share of cells that are '-' / blank, by event type (1.0 = always not applicable):")
display(share_na)

net = host["event_type"].isin(["network_connect", "network_accept"])
print("network events with a complete tuple (IPs, ports, protocol):",
      int((~((host.loc[net, ["src_ip", "dst_ip", "src_port", "dst_port", "protocol"]] == "").any(axis=1) |
              (host.loc[net, ["src_ip", "dst_ip", "protocol"]] == "-").any(axis=1))).sum()), "of", int(net.sum()))
print("non-network events carrying a tuple:",
      int(((host.loc[~net, ["src_ip", "dst_ip", "protocol"]] != "-").any(axis=1)).sum()))
print("IoT cells that are '-' or blank:", int(((iot == "-") | (iot == "")).sum().sum()))

# disk / memory values only where the event can produce them
fe = host["event_type"] == "file_event"
rd = host["file_operation"] == "read"; wr = host["file_operation"] == "write"
print("\nfile reads with disk_read_bytes > 0   :", f"{(num(host[rd], 'disk_read_bytes') > 0).mean() * 100:.1f}%")
print("file writes with disk_write_bytes > 0 :", f"{(num(host[wr], 'disk_write_bytes') > 0).mean() * 100:.1f}%")
print("network / memory events with any disk bytes:",
      int(((num(host[~fe & ~(host['event_type'] == 'process_start')], 'disk_read_bytes') > 0) |
           (num(host[~fe & ~(host['event_type'] == 'process_start')], 'disk_write_bytes') > 0)).sum()))
print("memory_alloc_bytes > 0 only on memory events:",
      bool(((num(host, "memory_alloc_bytes") > 0) == (host["event_type"] == "memory_event")).all()))

share of cells that are '-' / blank, by event type (1.0 = always not applicable):


,events,src_ip,dst_ip,src_port,dst_port,protocol,file_operation
event_type,,,,,,,
file_event,40555,1.0,1.0,1.0,1.0,1.0,0.0
memory_event,21212,1.0,1.0,1.0,1.0,1.0,1.0
network_accept,38583,0.0,0.0,0.0,0.0,0.0,1.0
network_connect,19395,0.0,0.0,0.0,0.0,0.0,1.0
process_start,19200,1.0,1.0,1.0,1.0,1.0,1.0


network events with a complete tuple (IPs, ports, protocol): 57978 of 57978
non-network events carrying a tuple: 0
IoT cells that are '-' or blank: 0

file reads with disk_read_bytes > 0   : 100.0%
file writes with disk_write_bytes > 0 : 100.0%


network / memory events with any disk bytes: 0
memory_alloc_bytes > 0 only on memory events: True


**Result:** the placeholders follow the event type exactly. `src_ip`, `dst_ip`, `src_port`, `dst_port` and `protocol` are
filled on **100 % of the 57,978 network events** and are `-` / blank on **100 %** of file, memory and `process_start`
events. `file_operation` is filled on
every file event and `-` on all others. File reads and writes carry disk bytes, network and memory events carry none, and
`memory_alloc_bytes` is positive only on memory events. The IoT file has no placeholders at all. So **nothing is missing
and nothing needs imputing**: the `-` means "not applicable".

**How the placeholders will be used (decided now, applied in Stage 4).** In the graph features, `-` in a categorical
column becomes its own category (**N/A**, one-hot encoded) and a blank port becomes N/A (a null plus an "is applicable"
flag). They are never filled with a number such as 0, because that would invent a port or an address that never existed.
The V6 v2 files are not changed.

## Step 5: Review every column

Each column is given a **role**, so it is clear what is used for what:

| Role | Meaning |
|---|---|
| **structure** | used to link events and build the graph (time, addresses, ports, process ids); not scaled as a feature |
| **identity** | a readable name for an entity |
| **categorical feature** | one-hot encoded in Stage 4 |
| **numeric feature** | scaled in Stage 4 |
| **component output** | a prediction made by Component 1 or 2 (its own, imperfect detection result) |
| **never a feature** | ground truth, kept in the file for choosing the training set and for evaluation |

In [6]:
# ==========================================
# STEP 5a: COLUMN ROLES
# ==========================================
HOST_ROLES = {
    "timestamp": "structure", "host_id": "identity", "host_ip": "structure", "process_id": "structure",
    "parent_process_id": "structure", "process_name": "categorical feature", "event_type": "categorical feature",
    "trace_mode": "component output", "trace_source": "categorical feature", "src_ip": "structure",
    "dst_ip": "structure", "src_port": "structure", "dst_port": "structure", "protocol": "categorical feature",
    "syscall_name": "categorical feature", "syscall_result": "categorical feature",
    "process_state": "categorical feature", "memory_alloc_bytes": "numeric feature",
    "file_operation": "categorical feature", "threat_confidence_score": "component output",
    "threat_state": "component output", "cpu_usage": "numeric feature", "disk_read_bytes": "numeric feature",
    "disk_write_bytes": "numeric feature", "label": "never a feature", "type": "never a feature"}
IOT_ROLES = {
    "timestamp": "structure", "iot_device_id": "identity", "src_ip": "structure", "dst_ip": "structure",
    "src_port": "structure", "dst_port": "structure", "protocol": "categorical feature",
    "service": "categorical feature", "flow_duration": "numeric feature", "src_bytes": "numeric feature",
    "dst_bytes": "numeric feature", "src_packets": "numeric feature", "dst_packets": "numeric feature",
    "iat_mean": "numeric feature", "iat_std": "numeric feature", "packet_size_mean": "numeric feature",
    "packet_size_std": "numeric feature", "connection_rate": "numeric feature",
    "confidence_score": "component output", "label": "never a feature", "predicted_class": "component output",
    "attack_type": "never a feature"}
assert list(HOST_ROLES) == HOST_COLS and list(IOT_ROLES) == IOT_COLS

roles = pd.concat([
    pd.DataFrame({"dataset": "host", "column": list(HOST_ROLES), "role": list(HOST_ROLES.values()),
                  "distinct_values": [host[c].nunique() for c in HOST_ROLES]}),
    pd.DataFrame({"dataset": "iot", "column": list(IOT_ROLES), "role": list(IOT_ROLES.values()),
                  "distinct_values": [iot[c].nunique() for c in IOT_ROLES]})], ignore_index=True)
display(roles.groupby(["dataset", "role"]).size().unstack(fill_value=0))
print("never-a-feature columns (kept in the file, excluded from the model input X):")
print("  host:", [c for c, r in HOST_ROLES.items() if r == "never a feature"])
print("  iot :", [c for c, r in IOT_ROLES.items() if r == "never a feature"])
roles.to_csv(Path(DATA_DIR) / "column_roles.csv", index=False)
print("saved", Path(DATA_DIR) / "column_roles.csv")

role,categorical feature,component output,identity,never a feature,numeric feature,structure
dataset,,,,,,
host,8,3,1,2,4,8
iot,2,2,1,2,10,5


never-a-feature columns (kept in the file, excluded from the model input X):
  host: ['label', 'type']
  iot : ['label', 'attack_type']
saved D:\GNN\column_roles.csv


**Result:** every one of the 26 + 22 columns has a role. `label` and `type` (host) and `label` and `attack_type` (IoT) are
**never features**: they stay in the files because they are needed to choose the all-normal training scenarios and to
evaluate the model (Precision, Recall, F1, FPR, ROC-AUC, attack-type breakdown). `attack_type` is Component 2's
*predicted* category, excluded from the input as the plan states. The Component 1 and 2 detection outputs
(`threat_*`, `trace_mode`, `confidence_score`, `predicted_class`) are legitimate graph features from Stage 4 onwards.

In [7]:
# ==========================================
# STEP 5b: REDUNDANCY SCAN (does one column fully determine another?)
# ==========================================
def determines(df, max_card=60):
    card = df.nunique()
    found = []
    for a in df.columns:
        if a == "timestamp" or card[a] > max_card or card[a] == 1:
            continue
        g = df.groupby(a)[[c for c in df.columns if c != a and c != "timestamp"]].nunique()
        found += [(a, c) for c in g.columns if (g[c] == 1).all() and card[c] > 1]
    return found

for name, df in (("host", host), ("iot", iot)):
    pairs = determines(df)
    both = sorted({tuple(sorted(p)) for p in pairs if (p[1], p[0]) in pairs})
    one_way = [p for p in pairs if (p[1], p[0]) not in pairs]
    print(f"{name}: columns that are 1:1 (identical information) :", both or "none")
    print(f"{name}: one column fully determines another          :", one_way or "none")
print("\nconstant columns:", [c for c in host.columns if host[c].nunique() == 1] + [c for c in iot.columns if iot[c].nunique() == 1] or "none")

host: columns that are 1:1 (identical information) : [('host_id', 'host_ip')]
host: one column fully determines another          : [('syscall_name', 'event_type'), ('syscall_name', 'file_operation'), ('type', 'label')]


iot: columns that are 1:1 (identical information) : [('iot_device_id', 'src_ip')]
iot: one column fully determines another          : [('attack_type', 'predicted_class')]



constant columns: none


**Result:** no constant columns. The scan finds a few relationships that are **kept on purpose**:

| Pair | Relationship | Why it stays |
|---|---|---|
| `host_id` ↔ `host_ip` | 1:1 (8 hosts) | `host_ip` is the node key and the join key to the IoT data; `host_id` is the readable name |
| `iot_device_id` ↔ `src_ip` | 1:1 (14 devices) | `src_ip` is the node key and join key; `iot_device_id` is the readable name |
| `syscall_name` → `event_type`, `file_operation` | one-way | kept as "redundant but safe" in the plan; the model sees a finer-grained category |
| `type` → `label`, `attack_type` → `predicted_class` | one-way | different roles: ground truth vs component prediction, both kept for evaluation |

These are documented design decisions, not oversights; any of them can be dropped by editing one list if a reviewer
prefers a smaller file.

## Step 6: Consistency of keys, processes and order

Checks that only need the two files themselves: the event key `(timestamp, host_id, process_id, event_type)` is unique, every process id
belongs to one process name (PID 1, `systemd`, exists once on every host), the processes that are their own parent are only `systemd` and a few daemons, long-lived services (30 or more events) are counted, and both
files are in time order. These properties are what the linking rules of Stage 3 rely on.

In [8]:
# ==========================================
# STEP 6: CONSISTENCY OF KEYS, PROCESSES AND ORDER
# ==========================================
print("duplicate (timestamp, host_id, process_id, event_type):", int(host.duplicated(["timestamp", "host_id", "process_id", "event_type"]).sum()))
print("process ids with more than one process name:", int((host.groupby("process_id")["process_name"].nunique() > 1).sum()))
multi = host.groupby("process_id")["host_id"].nunique()
print("process ids on more than one host:", {int(k): int(v) for k, v in multi[multi > 1].items()}, "(PID 1 = systemd, one per host)")
selfp = host[host["process_id"] == host["parent_process_id"]]
print("rows whose parent id equals their own id:", len(selfp), "| of these PID 1 (systemd):", int((selfp["process_id"] == "1").sum()),
      "| other daemons:", sorted(selfp.loc[selfp["process_id"] != "1", "process_name"].unique()))
ev = host.groupby("process_id").size()
print("process ids:", len(ev), "| long-lived services (30+ events):", int((ev >= 30).sum()),
      "| other process ids: median", float(ev[ev < 30].median()), "events, maximum", int(ev[ev < 30].max()))
ps = host[host["event_type"] == "process_start"]
print("process_start rows:", len(ps), "| with a parent id that is itself a process in the file:", f"{ps['parent_process_id'].isin(set(host['process_id'])).mean() * 100:.1f}%")
print("both files sorted by time:", bool(host_ts.is_monotonic_increasing), bool(iot_ts.is_monotonic_increasing))

duplicate (timestamp, host_id, process_id, event_type): 0
process ids with more than one process name: 0
process ids on more than one host: {1: 8} (PID 1 = systemd, one per host)
rows whose parent id equals their own id: 4224 | of these PID 1 (systemd): 3984 | other daemons: ['mysqld', 'redis-server', 'sshd']
process ids: 25997 | long-lived services (30+ events): 51 | other process ids: median 1.0 events, maximum 16
process_start rows: 19200 | with a parent id that is itself a process in the file: 87.0%


both files sorted by time: True True


**Result:** the keys and the process structure are consistent. There is **no duplicate event key**, and every process id has exactly one
process name. The only process id on more than one host is **PID 1** (`systemd`, once on each of the 8 hosts), which is normal; Stage 3
treats PID 1 as a service. 4,224 rows have a parent id equal to their own id: 3,984 are `systemd` and the rest are three daemons
(`mysqld`, `redis-server`, `sshd`) whose parent is not recorded; they are long-lived services and are not used as linking entities.
There are 25,997 process ids, 51 of them long-lived services (30 or more events); every other process has a median of 1 event and at
most 16. Of the 19,200 `process_start` rows, 87.0 % have a parent that is itself a process in the file (the rest are handled by the
fallback rule in Stage 3). Both files are in time order.

## Step 6b: Leakage check

A feature leaks if it alone reveals the label. The check below uses only the columns that Step 5a marks as **categorical** or
**numeric features** (never `label`, `type`, `attack_type`, and not the Component 1 / 2 prediction columns). For every
categorical feature it reports the accuracy of the best one-column rule (predict the majority label of each value) against
the share of the majority class; for every numeric feature it reports the ROC-AUC of the raw value (0.5 = no information).

In [9]:
# ==========================================
# STEP 6b: LEAKAGE CHECK (single-column predictiveness of the label)
# ==========================================
def rank_auc(x, y):
    r = pd.Series(x).rank(method="average").to_numpy()
    y = np.asarray(y, dtype=bool)
    n1, n0 = y.sum(), (~y).sum()
    return float((r[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))

rows = []
for name, df, roles_ in (("host", host, HOST_ROLES), ("iot", iot, IOT_ROLES)):
    y = (df["label"] == "1").to_numpy()
    base = max(y.mean(), 1 - y.mean())
    for c, r in roles_.items():
        if r == "categorical feature":
            acc = df.groupby(c)["label"].agg(lambda s: s.value_counts().iloc[0]).sum() / len(df)
            rows.append([name, c, r, round(float(acc), 3), round(float(base), 3), round(float(acc - base), 3)])
        elif r == "numeric feature":
            a = rank_auc(pd.to_numeric(df[c]), y)
            rows.append([name, c, r, round(a, 3), 0.5, round(abs(a - 0.5), 3)])
leak = pd.DataFrame(rows, columns=["dataset", "column", "role", "score", "reference", "gap"])
print("categorical: score = best one-column rule accuracy, reference = majority share | numeric: score = ROC-AUC, reference = 0.5")
display(leak.sort_values("gap", ascending=False).head(12))
print("largest gap, categorical:", leak[leak["role"] == "categorical feature"]["gap"].max(),
      "| numeric (|AUC - 0.5|):", leak[leak["role"] == "numeric feature"]["gap"].max())


categorical: score = best one-column rule accuracy, reference = majority share | numeric: score = ROC-AUC, reference = 0.5


,dataset,column,role,score,reference,gap
23,iot,connection_rate,numeric feature,0.659,0.5,0.159
19,iot,iat_mean,numeric feature,0.366,0.5,0.134
20,iot,iat_std,numeric feature,0.371,0.5,0.129
9,host,cpu_usage,numeric feature,0.594,0.5,0.094
21,iot,packet_size_mean,numeric feature,0.588,0.5,0.088
22,iot,packet_size_std,numeric feature,0.584,0.5,0.084
15,iot,src_bytes,numeric feature,0.575,0.5,0.075
16,iot,dst_bytes,numeric feature,0.573,0.5,0.073
17,iot,src_packets,numeric feature,0.553,0.5,0.053
18,iot,dst_packets,numeric feature,0.552,0.5,0.052


largest gap, categorical: 0.0 | numeric (|AUC - 0.5|): 0.159


**Result:** **no feature leaks the label on its own.** For every categorical feature the best one-column rule is no better than
always predicting the majority class (gap 0.0), so no value of any categorical feature points to an attack. The numeric
features are only weakly informative: the largest deviation from 0.5 is `connection_rate` (ROC-AUC 0.659), then
`iat_mean` (0.366) and `iat_std` (0.371) in the IoT flows; the host features are weaker (`cpu_usage` 0.594). The ground-truth
columns (`label`, `type`, `attack_type`) and the Component 1 / 2 prediction columns are not part of this check; the first are
never features, the second are legitimate features whose effect is tested separately in Stage 5 (model with and without them).

## Step 7: Decisions and hand-off to Stage 3

In [10]:
# ==========================================
# STEP 7: SUMMARY
# ==========================================
summary = pd.DataFrame([
    ["Rows", f"{len(host):,} host / {len(iot):,} iot", "no row removed or changed by this notebook"],
    ["Columns kept", "26 host / 22 iot", "as in the files"],
    ["Missing / invalid values", "none", "no duplicates, valid IPs / ports / scores, numeric columns parse"],
    ["Placeholders '-' / blank", "kept as they are", "'not applicable' by event type; never imputed; become an N/A category in Stage 4"],
    ["Timestamps", "DD/MM/YYYY HH:MM:SS", "parsed with an explicit format in every later stage (never opened in Excel)"],
    ["Ground truth", "label, type / label, attack_type", "never features; used only to pick normal scenarios and to evaluate"],
    ["Encoding and scaling", "not done here", "happens inside Stage 4 (graph construction), using column_roles.csv"],
], columns=["item", "state", "note"])
display(summary)

,item,state,note
0,Rows,"138,945 host / 59,415 iot",no row removed or changed by this notebook
1,Columns kept,26 host / 22 iot,as in the files
2,Missing / invalid values,none,"no duplicates, valid IPs / ports / scores, numeric columns parse"
3,Placeholders '-' / blank,kept as they are,'not applicable' by event type; never imputed; become an N/A category in Stage 4
4,Timestamps,DD/MM/YYYY HH:MM:SS,parsed with an explicit format in every later stage (never opened in Excel)
5,Ground truth,"label, type / label, attack_type",never features; used only to pick normal scenarios and to evaluate
6,Encoding and scaling,not done here,"happens inside Stage 4 (graph construction), using column_roles.csv"


**Result:** **no change to the dataset is needed.** The two V6 v2 files are final and are used exactly as they are by Stage 3
(relationship mapping). The only remaining "preparation" is the feature encoding (one-hot for categories, scaling for numbers),
which belongs to Stage 4 because it is part of building each graph.

## Limitations

| Limitation | How the notebook reduces it | What remains |
|---|---|---|
| Emulated data, not real TON_IoT | Uses the Component 1 / 2 output format and checks every rule | The checks are proven here; real data will have genuine missing values and mixed formats, so the same checks must be repeated and an imputation policy decided |
| Part of the data is synthetic follow-up activity added to small scenarios | Consistency checked in Step 6 (unique keys, one name and host per process, processes below the service limit) | It shows what the generator considers realistic follow-up activity, not recorded traffic; state this in the report |
| Scenario sizes were matched between normal and attack | Removes scenario size and duration as a shortcut; the shortcut audit in Stage 3b measures the remainder | The number of hosts per scenario can still separate the classes weakly and is checked again in Stage 5 |
| Redundant pairs kept (`host_id` ↔ `host_ip`, `iot_device_id` ↔ `src_ip`, `syscall_name`, ...) | Each is documented with a reason (Step 5b) | They add no information; they can be dropped with a one-line change if a smaller file is preferred |
| Column roles are a design decision | Roles are stored in `column_roles.csv` so they can be reviewed | The feature set may be revised after the first training results |
| Class balance (about 13 % attack host rows, 17 % iot flows) | Normal is the majority, as required | The model is trained on normal scenarios only, so the attack share matters only for testing |
| `attack_type` is a prediction, not ground truth | Marked as "never a feature" and kept for the evaluation breakdown | The true attack category is not exported |